# Does Regularization Shape Prunability? — experiment runner
Settings → Accelerator: **GPU T4 x2**, Internet: **On**.
**Add Input → your Kaggle dataset containing `cifar-10-python.tar.gz`** (md5 c58f30108f718f92721af3b95e74349a) to skip the 35-minute download.
Each 100-epoch ResNet-18 run takes ~25 min on a T4; two runs execute in parallel, one per GPU.
Use **Save Version → Save & Run All (Commit)** so outputs in `/kaggle/working` are kept after the session ends.

In [ ]:
import os, subprocess, glob, shutil
REPO = 'https://github.com/WHITELOTUS0/dropout-prunability.git'
if not os.path.exists('/kaggle/working/code'):
    subprocess.run(['git', 'clone', REPO, '/kaggle/working/code'], check=True)
os.chdir('/kaggle/working/code')
!git pull -q; nvidia-smi --query-gpu=name,memory.total --format=csv; python -c "import torch;print(torch.__version__, torch.cuda.device_count())"
# Use an attached Kaggle dataset copy of CIFAR-10 if present (the Toronto server downloads at ~75 kB/s, 35 min)
os.makedirs('data', exist_ok=True)
tars = glob.glob('/kaggle/input/**/cifar-10-python.tar.gz', recursive=True)
if tars and not os.path.exists('data/cifar-10-python.tar.gz'):
    shutil.copy(tars[0], 'data/cifar-10-python.tar.gz'); print('using dataset copy:', tars[0])
else:
    print('no attached CIFAR-10 tarball found; torchvision will download it (slow)' if not tars else 'tarball already present')

## Config — which runs this session should execute

In [ ]:
EPOCHS = 100
SEEDS  = '0'                 # e.g. '0' for the first pass, '1 2' later
GPU0_P = '0.0 0.1'           # dropout rates handled by GPU 0
GPU1_P = '0.3 0.5'           # dropout rates handled by GPU 1 (ignored if only one GPU)
RUN_FINETUNE = False         # E1 (one-shot vs iterative) — enable once dense runs exist
os.makedirs('runs', exist_ok=True)

## Pilot (B3): smoke-test the pipeline for 1 epoch before committing hours

In [ ]:
!python train.py --p 0.0 --seed 0 --epochs 1 --out runs_pilot && python prune_sweep.py --run runs_pilot/p0.0_s0 --skip_layerwise

## Dense training matrix + pruning sweeps (both GPUs in parallel)

In [ ]:
import torch, subprocess, time
ngpu = torch.cuda.device_count()
procs = [subprocess.Popen(['bash', 'run_matrix.sh', GPU0_P, SEEDS, str(EPOCHS), '0'])]
if ngpu > 1:
    procs.append(subprocess.Popen(['bash', 'run_matrix.sh', GPU1_P, SEEDS, str(EPOCHS), '1']))
else:
    procs[0].wait(); procs.append(subprocess.Popen(['bash', 'run_matrix.sh', GPU1_P, SEEDS, str(EPOCHS), '0']))
for p in procs: p.wait()
!tail -n 3 runs/train_*.log

## E1: one-shot vs iterative at matched 20-epoch budget (s ∈ {60, 80, 90}%)

In [ ]:
if RUN_FINETUNE:
    import glob, itertools
    jobs = [(r, s, sch) for r in sorted(glob.glob('runs/p*_s*')) for s in ['0.6', '0.8', '0.9'] for sch in ['oneshot', 'iterative']
            if not os.path.exists(f'{r}/ft_{sch}_s{s}.json')]
    # round-robin the jobs over GPUs
    queues = [jobs[i::max(ngpu,1)] for i in range(max(ngpu,1))]
    def run_queue(q, gpu):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
        cmd = ' && '.join(f'python finetune.py --run {r} --sparsity {s} --schedule {sch}' for r, s, sch in q) or 'true'
        return subprocess.Popen(['bash', '-c', cmd], env=env)
    ps = [run_queue(q, g) for g, q in enumerate(queues)]
    for p in ps: p.wait()

## Figures + tables

In [ ]:
!pip -q install tabulate
!python plots.py --runs runs --out figures
!ls figures
# bundle everything except checkpoints and data for download
!cd /kaggle/working && tar czf results_$(date +%Y%m%d_%H%M).tar.gz code/runs/*/*.json code/runs/*.log code/figures && ls -la /kaggle/working/*.tar.gz
from IPython.display import Image, display
for f in ['acc_vs_sparsity.png', 'random_vs_magnitude.png', 'weight_hist.png', 'training_curves.png']:
    if os.path.exists('figures/' + f): display(Image('figures/' + f))